In [172]:
import pandas as pd
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt

In [173]:
PROJECT_DIR = Path.cwd().resolve().parent
DATA_DIR = PROJECT_DIR / 'data'
SRC_DIR = PROJECT_DIR / 'src'
EXTRACT_DIR = PROJECT_DIR / 'processed' / 'extraction'
OUTPUT_DIR = PROJECT_DIR / 'processed' / 'transform'
REPORT_DIR = PROJECT_DIR / 'reports'

print('PROJECT_DIR:', PROJECT_DIR)
print('EXTRACT_DIR:', EXTRACT_DIR)
print('OUTPUT_DIR:', OUTPUT_DIR)

PROJECT_DIR: /Users/oneash/ONEASH_local/Coding/Delirium-Prediction/Parkinson
EXTRACT_DIR: /Users/oneash/ONEASH_local/Coding/Delirium-Prediction/Parkinson/processed/extraction
OUTPUT_DIR: /Users/oneash/ONEASH_local/Coding/Delirium-Prediction/Parkinson/processed/transform


## 데이터 로드

In [174]:
# all events (chart + lab)
all_events = pd.read_csv(EXTRACT_DIR / 'all_events_long.csv')

print(f"all_events shape: {all_events.shape}")
print(f"Columns: {all_events.columns.tolist()}")
all_events.head()

all_events shape: (795913, 12)
Columns: ['source_table', 'subject_id', 'hadm_id', 'stay_id', 'charttime', 'itemid', 'label', 'feature_name', 'type', 'value', 'valuenum', 'valueuom']


,source_table,subject_id,hadm_id,stay_id,charttime,itemid,label,feature_name,type,value,valuenum,valueuom
0,chartevents,14979348,22895491.0,39414466,2139-08-31 16:00:00,225664,Glucose finger stick (range 70-100),glucose,numeric,87,87.0,NaN
1,chartevents,14979348,22895491.0,39414466,2139-08-30 23:50:00,220179,Non Invasive Blood Pressure systolic,nibp_sbp,numeric,170,170.0,mmHg
2,chartevents,14979348,22895491.0,39414466,2139-08-30 23:50:00,220180,Non Invasive Blood Pressure diastolic,nibp_dbp,numeric,80,80.0,mmHg
3,chartevents,14979348,22895491.0,39414466,2139-08-30 23:50:00,220181,Non Invasive Blood Pressure mean,nibp_mbp,numeric,102,102.0,mmHg
4,chartevents,14979348,22895491.0,39414466,2139-08-30 23:51:00,220045,Heart Rate,heart_rate,numeric,93,93.0,bpm


In [175]:
# adm_pat_icu (icu stay info)
adm_pat_icu = pd.read_csv(EXTRACT_DIR / 'adm_pat_icu.csv')
print(f"adm_pat_icu shape: {adm_pat_icu.shape}")
print(f"Columns: {adm_pat_icu.columns.tolist()}")
adm_pat_icu.head()

adm_pat_icu shape: (974, 19)
Columns: ['subject_id', 'hadm_id', 'stay_id', 'first_careunit', 'last_careunit', 'intime', 'outtime', 'los', 'gender', 'anchor_age', 'anchor_year', 'anchor_year_group', 'dod', 'admittime', 'dischtime', 'deathtime', 'admission_type', 'race', 'icu_los_hours']


,subject_id,hadm_id,stay_id,first_careunit,last_careunit,intime,outtime,los,gender,anchor_age,anchor_year,anchor_year_group,dod,admittime,dischtime,deathtime,admission_type,race,icu_los_hours
0,10018328,23786647,31269608,Neuro Stepdown,Neuro Stepdown,2154-04-24 23:03:44,2154-05-02 15:55:21,7.702512,F,83,2154,2014 - 2016,2158-12-14,2154-04-24 03:15:00,2154-05-03 14:00:00,NaN,SURGICAL SAME DAY ADMISSION,WHITE,184.860278
1,10018328,28252562,37006782,Neuro Intermediate,Neuro Intermediate,2158-02-08 22:56:08,2158-02-13 23:58:23,5.043229,F,83,2154,2014 - 2016,2158-12-14,2158-02-08 22:55:00,2158-02-18 17:30:00,NaN,OBSERVATION ADMIT,WHITE,121.037500
2,10050755,20050796,37743005,Medical/Surgical Intensive Care Unit (MICU/SICU),Medical/Surgical Intensive Care Unit (MICU/SICU),2134-02-26 01:03:02,2134-02-26 04:27:45,0.142164,M,77,2126,2008 - 2010,2134-02-26,2134-02-25 18:41:00,2134-02-26 01:26:00,2134-02-26 01:26:00,OBSERVATION ADMIT,WHITE - RUSSIAN,3.411944
3,10052769,22087051,30336368,Neuro Surgical Intensive Care Unit (Neuro SICU),Neuro Surgical Intensive Care Unit (Neuro SICU),2124-06-16 20:18:49,2124-07-08 14:31:39,21.758912,M,78,2124,2020 - 2022,2124-07-09,2124-04-25 19:33:00,2124-07-09 09:43:00,2124-07-09 09:43:00,URGENT,UNKNOWN,522.213889
4,10052769,22087051,34171709,Medical/Surgical Intensive Care Unit (MICU/SICU),Medical/Surgical Intensive Care Unit (MICU/SICU),2124-05-25 15:48:47,2124-05-26 02:14:53,0.434792,M,78,2124,2020 - 2022,2124-07-09,2124-04-25 19:33:00,2124-07-09 09:43:00,2124-07-09 09:43:00,URGENT,UNKNOWN,10.435000


In [176]:
# Date columns
all_events['charttime'] = pd.to_datetime(all_events['charttime'], errors='coerce')
for col in ['intime', 'outtime', 'admittime', 'dischtime', 'deathtime']:
    adm_pat_icu[col] = pd.to_datetime(adm_pat_icu[col], errors='coerce')

## VALUE 변환 (문자열 → 숫자)

In [177]:
all_events.rename(columns={'value': 'value_str',
                           'valuenum': 'value'}, inplace=True)

In [178]:
name = 'gcs_eye'
display(all_events[all_events.feature_name == name].head())
all_events[all_events.feature_name == name].value_str.unique()

,source_table,subject_id,hadm_id,stay_id,charttime,itemid,label,feature_name,type,value_str,value,valueuom
17,chartevents,14979348,22895491.0,39414466,2139-09-01 08:00:00,220739,GCS - Eye Opening,gcs_eye,ordinal,NaN,1.0,NaN
39,chartevents,14979348,22895491.0,39414466,2139-09-01 10:00:00,220739,GCS - Eye Opening,gcs_eye,ordinal,NaN,1.0,NaN
48,chartevents,14979348,22895491.0,39414466,2139-09-01 11:45:00,220739,GCS - Eye Opening,gcs_eye,ordinal,NaN,1.0,NaN
59,chartevents,14979348,22895491.0,39414466,2139-09-01 12:00:00,220739,GCS - Eye Opening,gcs_eye,ordinal,NaN,1.0,NaN
76,chartevents,14979348,22895491.0,39414466,2139-09-01 16:00:00,220739,GCS - Eye Opening,gcs_eye,ordinal,NaN,1.0,NaN


<ArrowStringArray>
[nan, 'Spontaneously', 'To Speech', 'To Pain']
Length: 4, dtype: str

In [179]:
all_events[all_events.feature_name == name].info()

<class 'pandas.DataFrame'>
Index: 23911 entries, 17 to 688750
Data columns (total 12 columns):
 #   Column        Non-Null Count  Dtype         
---  ------        --------------  -----         
 0   source_table  23911 non-null  str           
 1   subject_id    23911 non-null  int64         
 2   hadm_id       23911 non-null  float64       
 3   stay_id       23911 non-null  int64         
 4   charttime     23911 non-null  datetime64[us]
 5   itemid        23911 non-null  int64         
 6   label         23911 non-null  str           
 7   feature_name  23911 non-null  str           
 8   type          23911 non-null  str           
 9   value_str     21568 non-null  str           
 10  value         23911 non-null  float64       
 11  valueuom      0 non-null      str           
dtypes: datetime64[us](1), float64(2), int64(3), str(6)
memory usage: 3.6 MB


In [180]:
# 문자열/범주형 -> 숫자 변환

# Delirium assessment: Positive=1, Negative=0, UTA/other=NaN
all_events.loc[all_events['value_str'].astype('string').str.lower().eq('positive'), 'value'] = 1
all_events.loc[all_events['value_str'].astype('string').str.lower().eq('negative'), 'value'] = 0

# RASS text values
all_events.loc[all_events['value_str'] == '"+4 Combative' , 'value']        = 4
all_events.loc[all_events['value_str'] == '+3 Pulls or removes tube(s) or catheter(s); aggressive', 'value'] = 3
all_events.loc[all_events['value_str'] == '"+2 Frequent nonpurposeful movement', 'value']                    = 2
all_events.loc[all_events['value_str'] == '"+1 Anxious', 'value']           = 1
all_events.loc[all_events['value_str'] == ' 0  Alert and calm', 'value']    = 0
all_events.loc[all_events['value_str'] == '-1 Awakens to voice (eye opening/contact) > 10 sec', 'value'] = -1
all_events.loc[all_events['value_str'] == '"-2 Light sedation' , 'value']   = -2
all_events.loc[all_events['value_str'] == '"-3 Moderate sedation', 'value'] = -3
all_events.loc[all_events['value_str'] == '"-4 Deep sedation', 'value']     = -4
all_events.loc[all_events['value_str'] == '"-5 Unarousable' , 'value']      = -5


## 단위 변환

In [181]:
if 'valueuom_original' not in all_events.columns:
    all_events['valueuom_original'] = all_events['valueuom']

def fahr_to_celsius(temp_fahr):
    return (temp_fahr - 32) * 5 / 9

# Temperature: Fahrenheit -> Celsius
temp_f_mask = all_events['label'].eq('Temperature Fahrenheit')
all_events.loc[temp_f_mask, 'value'] = fahr_to_celsius(all_events.loc[temp_f_mask, 'value'])
all_events.loc[all_events['feature_name'].eq('temperature'), 'valueuom'] = 'C'

# Weight: lbs -> kg
weight_lbs_mask = all_events['label'].eq('Admission Weight (lbs.)')
all_events.loc[weight_lbs_mask, 'value'] = all_events.loc[weight_lbs_mask, 'value'] * 0.453592
all_events.loc[all_events['feature_name'].eq('weight'), 'valueuom'] = 'kg'

# Height: inch -> cm
height_inch_mask = all_events['label'].eq('Height')
all_events.loc[height_inch_mask, 'value'] = all_events.loc[height_inch_mask, 'value'] * 2.54
all_events.loc[all_events['feature_name'].eq('height'), 'valueuom'] = 'cm'


In [182]:
all_events.to_csv(OUTPUT_DIR / 'all_events_filtered.csv', index=False)


## 시간 계산 (ICU 입실 기준)

In [183]:
# length of stay in icu (hours)
adm_pat_icu['los_hours'] = ((adm_pat_icu['outtime'] - adm_pat_icu['intime']).dt.total_seconds() / 3600).astype(int)

In [184]:
patient_cols = [
    'stay_id', 'subject_id', 'hadm_id', 'anchor_age', 'gender', 'los_hours',
    'admission_type', 'race', 'hospital_expire_flag', 'intime', 'outtime'
]
patient_cols = [c for c in patient_cols if c in adm_pat_icu.columns]
merge_keys = [c for c in ['stay_id', 'subject_id', 'hadm_id'] if c in all_events.columns and c in patient_cols]
all_events = all_events.merge(adm_pat_icu[patient_cols], on=merge_keys, how='inner')

# ICU 입실 후 경과시간 계산
all_events['hours'] = ((all_events['charttime'] - all_events['intime']).dt.total_seconds() / 3600.0)

# Keep events within ICU stay
all_events = all_events[(all_events['hours'] >= 0) & (all_events['charttime'] <= all_events['outtime'])].copy()

# 60-minute binning
all_events['bin'] = all_events['hours'].astype(int)

## 60분 비닝 및 피봇 (최종 시계열)

In [185]:
# 시간 bin별 기본 정보 (나이, 성별, LOS, 입원 유형 등 시간별 행에 반복)
base_agg = {
    'hours': 'max',
    'anchor_age': 'max',
    'gender': 'first',
    'los_hours': 'max',
    'subject_id': 'first',
    'hadm_id': 'first',
    'race': 'first',
}
base_agg = {col: agg for col, agg in base_agg.items() if col in all_events.columns}
base_info = all_events.groupby(['stay_id', 'bin']).agg(base_agg).reset_index()
base_info.rename(columns={'anchor_age': 'age'}, inplace=True)
base_info.head(10)

,stay_id,bin,hours,age,gender,los_hours,subject_id,hadm_id,race
0,30008046,0,0.669722,75,M,99,19448158,28451027.0,WHITE
1,30008046,1,1.686389,75,M,99,19448158,28451027.0,WHITE
2,30008046,2,2.403056,75,M,99,19448158,28451027.0,WHITE
3,30008046,4,4.403056,75,M,99,19448158,28451027.0,WHITE
4,30008046,6,6.403056,75,M,99,19448158,28451027.0,WHITE
5,30008046,8,8.403056,75,M,99,19448158,28451027.0,WHITE
6,30008046,10,10.403056,75,M,99,19448158,28451027.0,WHITE
7,30008046,11,11.336389,75,M,99,19448158,28451027.0,WHITE
8,30008046,12,12.403056,75,M,99,19448158,28451027.0,WHITE
9,30008046,14,14.403056,75,M,99,19448158,28451027.0,WHITE


In [186]:
# 이벤트 long-format 데이터를 stay_id-bin 단위 wide-format으로 변환
# 같은 시간 bin에 여러 값이 있으면 max 값 사용
pivoted = all_events.pivot_table(
    index=['stay_id', 'bin'],
    columns='feature_name',
    values='value',
    aggfunc='max'
).reset_index()
pivoted.columns.name = None
pivoted.head(10)

,stay_id,bin,BUN,abp_dbp,abp_mbp,abp_sbp,albumin,alt,ast,bicarbonate,...,pt,ptt,rass,respiratory_rate,sao2,sodium,spo2,temperature,wbc,weight
0,30008046,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,0.0,29.0,NaN,NaN,97.0,37.111111,NaN,NaN
1,30008046,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,81.147609
2,30008046,2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,0.0,25.0,NaN,NaN,97.0,36.666667,NaN,NaN
3,30008046,4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,23.0,NaN,NaN,100.0,NaN,NaN,NaN
4,30008046,6,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,19.0,NaN,NaN,99.0,36.444444,NaN,NaN
5,30008046,8,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,19.0,NaN,NaN,100.0,NaN,NaN,NaN
6,30008046,10,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,22.0,NaN,NaN,100.0,NaN,NaN,NaN
7,30008046,11,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,36.333333,NaN,NaN
8,30008046,12,23.0,NaN,NaN,NaN,NaN,NaN,NaN,31.0,...,NaN,NaN,NaN,22.0,NaN,139.0,100.0,NaN,5.3,NaN
9,30008046,14,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,-1.0,20.0,NaN,NaN,99.0,36.666667,NaN,NaN


In [187]:
# 기본 정보와 pivot 결과를 결합
timeseries = base_info.merge(pivoted, on=['stay_id', 'bin'], how='left')
timeseries = timeseries.sort_values(['stay_id', 'bin']).reset_index(drop=True)

print("\n=== Final Timeseries ===")
print(f"Unique stay_ids: {timeseries['stay_id'].nunique()}")
print(f"Total rows: {len(timeseries)}")
print(f"\nColumns: {timeseries.columns.tolist()}")
display(timeseries.head(10))
timeseries.to_csv(OUTPUT_DIR / 'all_events_timeseries.csv', index=False)


=== Final Timeseries ===
Unique stay_ids: 974
Total rows: 86210

Columns: ['stay_id', 'bin', 'hours', 'age', 'gender', 'los_hours', 'subject_id', 'hadm_id', 'race', 'BUN', 'abp_dbp', 'abp_mbp', 'abp_sbp', 'albumin', 'alt', 'ast', 'bicarbonate', 'bicarbonate_gas', 'bilirubin_direct', 'bilirubin_indirect', 'bilirubin_total', 'calcium_free', 'chloride', 'creatinine', 'delirium_assessment', 'gcs_eye', 'gcs_motor', 'gcs_verbal', 'glucose', 'heart_rate', 'height', 'hematocrit', 'hemoglobin', 'inr_pt', 'lactate', 'magnesium', 'nibp_dbp', 'nibp_mbp', 'nibp_sbp', 'pco2', 'ph', 'phosphate', 'platelet_count', 'po2', 'potassium', 'pt', 'ptt', 'rass', 'respiratory_rate', 'sao2', 'sodium', 'spo2', 'temperature', 'wbc', 'weight']


,stay_id,bin,hours,age,gender,los_hours,subject_id,hadm_id,race,BUN,...,pt,ptt,rass,respiratory_rate,sao2,sodium,spo2,temperature,wbc,weight
0,30008046,0,0.669722,75,M,99,19448158,28451027.0,WHITE,NaN,...,NaN,NaN,0.0,29.0,NaN,NaN,97.0,37.111111,NaN,NaN
1,30008046,1,1.686389,75,M,99,19448158,28451027.0,WHITE,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,81.147609
2,30008046,2,2.403056,75,M,99,19448158,28451027.0,WHITE,NaN,...,NaN,NaN,0.0,25.0,NaN,NaN,97.0,36.666667,NaN,NaN
3,30008046,4,4.403056,75,M,99,19448158,28451027.0,WHITE,NaN,...,NaN,NaN,NaN,23.0,NaN,NaN,100.0,NaN,NaN,NaN
4,30008046,6,6.403056,75,M,99,19448158,28451027.0,WHITE,NaN,...,NaN,NaN,NaN,19.0,NaN,NaN,99.0,36.444444,NaN,NaN
5,30008046,8,8.403056,75,M,99,19448158,28451027.0,WHITE,NaN,...,NaN,NaN,NaN,19.0,NaN,NaN,100.0,NaN,NaN,NaN
6,30008046,10,10.403056,75,M,99,19448158,28451027.0,WHITE,NaN,...,NaN,NaN,NaN,22.0,NaN,NaN,100.0,NaN,NaN,NaN
7,30008046,11,11.336389,75,M,99,19448158,28451027.0,WHITE,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,36.333333,NaN,NaN
8,30008046,12,12.403056,75,M,99,19448158,28451027.0,WHITE,23.0,...,NaN,NaN,NaN,22.0,NaN,139.0,100.0,NaN,5.3,NaN
9,30008046,14,14.403056,75,M,99,19448158,28451027.0,WHITE,NaN,...,NaN,NaN,-1.0,20.0,NaN,NaN,99.0,36.666667,NaN,NaN


## STEP 6: Delirium assessment 양성/음성 분류

In [ ]:
timeseries = pd.read_csv(OUTPUT_DIR / 'all_events_timeseries.csv')

In [ ]:
# 섬망 평가 feature를 Delirium outcome 컬럼으로 정리합니다.
# 평가가 없는 시간대는 NaN으로 남겨 assessment 시점만 outcome으로 사용합니다.
if 'delirium_assessment' in timeseries.columns:
    timeseries['Delirium'] = timeseries['delirium_assessment']
    timeseries.drop(columns=['delirium_assessment'], inplace=True)
else:
    timeseries['Delirium'] = np.nan

print("Delirium value counts (NaN excluded):")
print(timeseries['Delirium'].value_counts())
print(f"NaN count: {timeseries['Delirium'].isna().sum():,}")


In [ ]:
# 약물 및 처치/장치 추출 결과를 불러옵니다.
# 시간 컬럼을 datetime으로 정리해 노출 구간 계산에 사용할 수 있게 합니다.
# Load medication/procedure extraction outputs and attach binary exposure columns.
# This does not rebuild all_events; it only transforms dedicated extraction outputs.
med_events_path = EXTRACT_DIR / 'medication_events.csv'
procedure_events_path = EXTRACT_DIR / 'procedure_selected.csv'
med_events = pd.read_csv(med_events_path) if med_events_path.exists() else pd.DataFrame()
procedure_events = pd.read_csv(procedure_events_path) if procedure_events_path.exists() else pd.DataFrame()

for col in ['charttime', 'intime', 'outtime']:
    if len(med_events) and col in med_events.columns:
        med_events[col] = pd.to_datetime(med_events[col], errors='coerce')
for col in ['starttime', 'endtime', 'intime', 'outtime']:
    if len(procedure_events) and col in procedure_events.columns:
        procedure_events[col] = pd.to_datetime(procedure_events[col], errors='coerce')

print('med_events:', med_events.shape)
print('procedure_events:', procedure_events.shape)


In [ ]:
# eMAR 약물 투약 이벤트를 시간 bin별 binary flag로 펼칩니다.
# 투약 시점과 8시간 lookback을 반영해 해당 bin에 1로 표시합니다.
MED_LOOKBACK_HOURS = 8
med_flags = pd.DataFrame(columns=['stay_id', 'bin'])

if len(med_events):
    med = med_events.copy()
    if 'intime' not in med.columns or 'outtime' not in med.columns:
        med_patient_cols = [c for c in ['stay_id', 'subject_id', 'hadm_id', 'intime', 'outtime'] if c in adm_pat_icu.columns]
        med_merge_keys = [c for c in ['stay_id', 'subject_id', 'hadm_id'] if c in med.columns and c in med_patient_cols]
        med = med.merge(adm_pat_icu[med_patient_cols], on=med_merge_keys, how='inner')

    med['intime'] = pd.to_datetime(med['intime'], errors='coerce')
    med['outtime'] = pd.to_datetime(med['outtime'], errors='coerce')
    med['event_time_use'] = pd.to_datetime(med['charttime'], errors='coerce')
    med = med[med['event_time_use'].notna() & med['intime'].notna() & med['outtime'].notna()].copy()
    med['event_bin'] = np.floor((med['event_time_use'] - med['intime']).dt.total_seconds() / 3600).astype('int64').clip(lower=0)
    med['max_bin'] = np.ceil((med['outtime'] - med['intime']).dt.total_seconds() / 3600).astype('int64')

    expanded = []
    for row_ in med[['stay_id', 'feature_name', 'event_bin', 'max_bin']].itertuples(index=False):
        exposure_start = max(0, int(row_.event_bin))
        exposure_end = min(int(row_.max_bin), int(row_.event_bin) + MED_LOOKBACK_HOURS - 1)
        if exposure_end >= exposure_start:
            expanded.append(pd.DataFrame({'stay_id': row_.stay_id, 'bin': np.arange(exposure_start, exposure_end + 1, dtype='int64'), 'medication_feature_name': row_.feature_name, 'value': 1}))

    if expanded:
        med_expanded = pd.concat(expanded, ignore_index=True).drop_duplicates(['stay_id', 'bin', 'medication_feature_name'])
        med_flags = med_expanded.pivot_table(index=['stay_id', 'bin'], columns='medication_feature_name', values='value', aggfunc='max', fill_value=0).reset_index()
        med_flags.columns.name = None

print('med_flags:', med_flags.shape)


In [ ]:
# 처치/장치 노출을 시간 bin별 binary flag로 펼칩니다.
# 이벤트 구간이 겹치는 모든 시간 bin에 해당 procedure feature를 1로 표시합니다.
# Procedure/device: interval overlap with current hour
procedure_flags = pd.DataFrame(columns=['stay_id', 'bin'])

if len(procedure_events):
    proc = procedure_events.copy()
    proc['start_use'] = proc['starttime']
    proc['end_use'] = proc['endtime'].fillna(proc['starttime'])
    proc['start_bin'] = np.floor((proc['start_use'] - proc['intime']).dt.total_seconds() / 3600).astype('int64').clip(lower=0)
    proc['end_bin'] = np.ceil((proc['end_use'] - proc['intime']).dt.total_seconds() / 3600).astype('int64')
    proc['max_bin'] = np.ceil((proc['outtime'] - proc['intime']).dt.total_seconds() / 3600).astype('int64')
    proc['end_bin'] = proc[['end_bin', 'max_bin']].min(axis=1)

    expanded = []
    for row_ in proc[['stay_id', 'feature_name', 'start_bin', 'end_bin']].itertuples(index=False):
        if pd.isna(row_.feature_name):
            continue
        start_bin = max(0, int(row_.start_bin))
        end_bin = max(start_bin, int(row_.end_bin))
        expanded.append(pd.DataFrame({'stay_id': row_.stay_id, 'bin': np.arange(start_bin, end_bin + 1, dtype='int64'), 'procedure_feature_name': row_.feature_name, 'value': 1}))

    if expanded:
        procedure_expanded = pd.concat(expanded, ignore_index=True).drop_duplicates(['stay_id', 'bin', 'procedure_feature_name'])
        procedure_flags = procedure_expanded.pivot_table(index=['stay_id', 'bin'], columns='procedure_feature_name', values='value', aggfunc='max', fill_value=0).reset_index()
        procedure_flags.columns.name = None

print('procedure_flags:', procedure_flags.shape)


In [ ]:
# timeseries에 약물/처치 binary flag를 병합합니다.
# 결측 노출값은 0으로 채우고 모든 binary feature를 0/1 정수형으로 정리합니다.
timeseries = timeseries.merge(med_flags, on=['stay_id', 'bin'], how='left')
timeseries = timeseries.merge(procedure_flags, on=['stay_id', 'bin'], how='left')

med_binary_cols = [c for c in med_flags.columns if c not in ['stay_id', 'bin']]
procedure_binary_cols = [c for c in timeseries.columns if c.startswith('procedure_')]
binary_cols = sorted(set([c for c in med_binary_cols + procedure_binary_cols if c in timeseries.columns]))

timeseries[binary_cols] = timeseries[binary_cols].fillna(0)
for col in binary_cols:
    timeseries[col] = (timeseries[col] > 0).astype('int8')

print(f"Binary columns: {binary_cols}")
print(f"Final columns ({len(timeseries.columns)}): {timeseries.columns.tolist()}")


In [ ]:
# 전체 시간별 timeseries를 저장합니다.
# Delirium은 평가 시점에만 0/1이고 나머지 시간은 NaN으로 유지됩니다.
# Save all timeseries. Delirium is 0/1 only at assessment times; otherwise NaN.
timeseries.to_csv(OUTPUT_DIR / 'all_timeseries.csv', index=False)
print(f"Saved: all_timeseries.csv ({len(timeseries):,} rows, {timeseries['stay_id'].nunique()} stays)")


## STEP 7: Imputation

In [ ]:
# 저장된 전체 timeseries를 다시 불러와 결측 처리 단계를 시작합니다.
timeseries = pd.read_csv(OUTPUT_DIR / 'all_timeseries.csv')


In [ ]:
# 약물 및 장치 binary 컬럼의 결측값을 0으로 채웁니다.
# 관찰되지 않은 노출은 해당 시간 bin에서 미노출로 해석합니다.
medication_feature_cols = []
med_events_path = EXTRACT_DIR / 'medication_events.csv'
if med_events_path.exists():
    medication_feature_cols = pd.read_csv(med_events_path, usecols=['feature_name'])['feature_name'].dropna().unique().tolist()

zero_fill_cols = [c for c in timeseries.columns if c.startswith('procedure_')]
zero_fill_cols += [c for c in medication_feature_cols if c in timeseries.columns]
zero_fill_cols = sorted(set(zero_fill_cols))

for col in zero_fill_cols:
    before_null = timeseries[col].isnull().sum()
    timeseries[col] = timeseries[col].fillna(0).astype('int8')
    print(f"  {col}: {before_null:,} NaN -> 0  (filled {before_null:,})")

print("\n7-1 done: medication/device zero-fill")


In [ ]:
# 체중과 키는 같은 ICU stay 안에서 앞/뒤 방향으로 채웁니다.
# 비교적 정적인 변수라 stay 내 관측값을 전체 시간축에 확장합니다.
static_fill_cols = ['weight', 'height']
static_fill_cols = [c for c in static_fill_cols if c in timeseries.columns]

for col in static_fill_cols:
    before_null = timeseries[col].isnull().sum()
    timeseries[col] = timeseries.groupby('stay_id')[col].transform(lambda v: v.ffill())
    timeseries[col] = timeseries.groupby('stay_id')[col].transform(lambda v: v.bfill())
    after_null = timeseries[col].isnull().sum()
    print(f"  {col}: {before_null:,} -> {after_null:,} NaN  (filled {before_null - after_null:,})")

print("\n7-2 done: weight, height patient-wise ffill + bfill")


In [ ]:
# 활력징후는 같은 stay 안에서 제한된 forward-fill만 적용합니다.
# 오래된 값이 과도하게 전파되지 않도록 최대 4시간까지만 채웁니다.
forward_fill_cols = [
    'heart_rate', 'respiratory_rate', 'temperature', 'spo2', 'sao2',
    'nibp_mbp', 'abp_mbp', 'nibp_sbp', 'abp_sbp', 'nibp_dbp', 'abp_dbp'
]
forward_fill_cols = [c for c in forward_fill_cols if c in timeseries.columns]
FFILL_LIMIT_HOURS = 4

print("7-3. Limited forward-fill (patient-wise, V/S only):")
for col in forward_fill_cols:
    before_null = timeseries[col].isnull().sum()
    timeseries[col] = timeseries.groupby('stay_id')[col].transform(lambda v: v.ffill(limit=FFILL_LIMIT_HOURS))
    after_null = timeseries[col].isnull().sum()
    filled = before_null - after_null
    if filled > 0:
        print(f"  {col}: {before_null:,} -> {after_null:,} NaN  (filled {filled:,})")

print(f"\n7-3 done: V/S forward-fill up to {FFILL_LIMIT_HOURS} hours")


In [ ]:
# lab 변수는 시간별로 강제 보간하지 않습니다.
# 최종 8시간 window 집계에서 window 내 최신 관측값을 사용합니다.
lab_cols = [
    'anion_gap', 'bicarbonate', 'bicarbonate_gas', 'calcium_free', 'chloride',
    'creatinine', 'glucose', 'magnesium', 'phosphate', 'potassium', 'sodium',
    'BUN', 'hematocrit', 'hemoglobin', 'platelet_count', 'wbc', 'alt', 'ast',
    'bilirubin_total', 'bilirubin_direct', 'bilirubin_indirect', 'albumin',
    'lactate', 'ph', 'pco2', 'po2', 'inr_pt', 'pt', 'ptt'
]
lab_cols = [c for c in lab_cols if c in timeseries.columns]
print(f"Lab columns kept without hourly fill: {lab_cols}")


In [ ]:
# 결측 처리 정책 적용 후 변수별 결측률을 확인합니다.
# Delirium은 평가 시점만 outcome이므로 별도로 평가 행 비율을 출력합니다.
# ========================================================================
# 7-5. Missingness after imputation policy
# ========================================================================
exclude_cols = ['stay_id', 'bin', 'hours', 'age', 'gender', 'los_hours', 'subject_id', 'hadm_id', 'Delirium']
feature_cols = [c for c in timeseries.columns if c not in exclude_cols]

percent_missing_after = timeseries[feature_cols].isnull().sum() * 100 / len(timeseries)
missing_df_after = pd.DataFrame({
    'column': feature_cols,
    'after_pct': percent_missing_after.values,
}).sort_values('after_pct', ascending=False).reset_index(drop=True)

print("=== Missingness after imputation policy ===")
print(missing_df_after.to_string())

assessed = timeseries['Delirium'].notna().sum()
total = len(timeseries)
print("\n=== Delirium assessment (separate) ===")
print(f"Assessment rows: {assessed:,} / {total:,} ({assessed/total*100:.2f}%)")
print(f"Unassessed rows (NaN): {total - assessed:,} ({(total-assessed)/total*100:.2f}%)")


In [ ]:
# 결측 처리 정책이 반영된 timeseries를 저장합니다.
# Save imputed data
timeseries.to_csv(OUTPUT_DIR / 'timeseries_imputed.csv', index=False)
print(f"Saved: timeseries_imputed.csv ({len(timeseries):,} rows, {timeseries['stay_id'].nunique()} stays)")


## STEP 8: Delirium assessment 평가 시점 기준 8시간 윈도우 집계

각 Delirium assessment 평가 시점으로부터 이전 8시간(assessment_bin-7 ~ assessment_bin) 데이터를 하나의 row로 집계합니다.

In [ ]:
# 저장된 imputed timeseries를 불러와 최종 assessment-window 데이터셋을 만듭니다.
timeseries = pd.read_csv(OUTPUT_DIR / 'timeseries_imputed.csv')


In [ ]:
def _nunique_if_exists(df, col):
    return df[col].nunique() if col in df.columns else np.nan

def build_cohort_counts(step, cohort_df, events_df):
    stay_ids = set(cohort_df['stay_id']) if 'stay_id' in cohort_df.columns else set()
    if 'stay_id' in events_df.columns:
        event_rows = int(events_df['stay_id'].isin(stay_ids).sum())
        event_stays = int(events_df.loc[events_df['stay_id'].isin(stay_ids), 'stay_id'].nunique())
    else:
        event_rows = np.nan
        event_stays = np.nan

    return {
        'step': step,
        'n_stays': int(_nunique_if_exists(cohort_df, 'stay_id')),
        'n_subjects': int(_nunique_if_exists(cohort_df, 'subject_id')),
        'n_hadm': int(_nunique_if_exists(cohort_df, 'hadm_id')),
        'event_rows': event_rows,
        'event_stays': event_stays,
    }

if 'icu_los_hours' not in adm_pat_icu.columns:
    adm_pat_icu['icu_los_hours'] = (adm_pat_icu['outtime'] - adm_pat_icu['intime']).dt.total_seconds() / 3600

adm_pat_icu_all = adm_pat_icu.copy()
print(f"All ICU stays kept for transform: {adm_pat_icu_all['stay_id'].nunique():,}")
print(f"All event rows kept for transform: {len(all_events):,}")


In [ ]:
print(f"adm_pat_icu for transform: {adm_pat_icu.shape}")
print(f"all_events for transform: {all_events.shape}")


In [ ]:
# Delirium 평가가 있는 시간 bin을 outcome 시점으로 식별합니다.
# Inclusion/exclusion criteria는 이 단계에서 순서대로 적용하고 cohort attrition 파일로 저장합니다.
# Identify Delirium assessment times
if 'build_cohort_counts' not in globals():
    def _nunique_if_exists(df, col):
        return df[col].nunique() if col in df.columns else np.nan

    def build_cohort_counts(step, cohort_df, events_df):
        stay_ids = set(cohort_df['stay_id']) if 'stay_id' in cohort_df.columns else set()
        if 'stay_id' in events_df.columns:
            event_rows = int(events_df['stay_id'].isin(stay_ids).sum())
            event_stays = int(events_df.loc[events_df['stay_id'].isin(stay_ids), 'stay_id'].nunique())
        else:
            event_rows = np.nan
            event_stays = np.nan

        return {
            'step': step,
            'n_stays': int(_nunique_if_exists(cohort_df, 'stay_id')),
            'n_subjects': int(_nunique_if_exists(cohort_df, 'subject_id')),
            'n_hadm': int(_nunique_if_exists(cohort_df, 'hadm_id')),
            'event_rows': event_rows,
            'event_stays': event_stays,
        }

assessments_all_raw = timeseries[timeseries['Delirium'].notna()][['stay_id', 'bin', 'Delirium']].copy()
assessments_all_raw.rename(columns={'bin': 'assessment_bin'}, inplace=True)

print(f"\nDelirium assessment rows before cohort filter: {len(assessments_all_raw):,}")
print(f"  - Unique stays: {assessments_all_raw['stay_id'].nunique()}")

adm_for_cohort = adm_pat_icu.copy()
for col in ['intime', 'outtime']:
    if col in adm_for_cohort.columns:
        adm_for_cohort[col] = pd.to_datetime(adm_for_cohort[col], errors='coerce')
if 'icu_los_hours' not in adm_for_cohort.columns:
    adm_for_cohort['icu_los_hours'] = (adm_for_cohort['outtime'] - adm_for_cohort['intime']).dt.total_seconds() / 3600

all_events_for_attrition = pd.DataFrame()
all_events_filtered_path = OUTPUT_DIR / 'all_events_filtered.csv'
if all_events_filtered_path.exists():
    all_events_for_attrition = pd.read_csv(all_events_filtered_path, usecols=lambda c: c == 'stay_id')

cohort_flow_rows = []
cohort = adm_for_cohort.copy()
cohort_flow_rows.append(build_cohort_counts('0. All ICU stays from extraction', cohort, all_events_for_attrition))

cohort = cohort[cohort['anchor_age'] >= 18].copy()
cohort_flow_rows.append(build_cohort_counts('1. Adult patients (anchor_age >= 18)', cohort, all_events_for_attrition))

cohort = cohort[cohort['intime'].notna() & cohort['outtime'].notna()].copy()
cohort_flow_rows.append(build_cohort_counts('2. Valid ICU intime/outtime', cohort, all_events_for_attrition))

cohort = cohort[cohort['icu_los_hours'] > 0].copy()
cohort_flow_rows.append(build_cohort_counts('3. Positive ICU LOS', cohort, all_events_for_attrition))

cohort = cohort[cohort['icu_los_hours'] >= 8].copy()
cohort_flow_rows.append(build_cohort_counts('4. ICU LOS >= 8 hours', cohort, all_events_for_attrition))

cohort_8hrs = cohort.copy()
cohort_8hr_stay_ids = set(cohort_8hrs['stay_id'])
cohort_8hrs.to_csv(OUTPUT_DIR / 'adm_pat_icu_8hrs.csv', index=False)

if all_events_filtered_path.exists():
    all_events_filtered_for_save = pd.read_csv(all_events_filtered_path)
    all_events_8hrs = all_events_filtered_for_save[all_events_filtered_for_save['stay_id'].isin(cohort_8hr_stay_ids)].copy()
    all_events_8hrs.to_csv(OUTPUT_DIR / 'all_events_8hrs.csv', index=False)
    print(f"all_events_8hrs saved: {all_events_8hrs.shape}")

assessments_all = assessments_all_raw[assessments_all_raw['stay_id'].isin(cohort_8hr_stay_ids)].copy()
assessment_stay_ids = set(assessments_all['stay_id'])
assessment_cohort = cohort_8hrs[cohort_8hrs['stay_id'].isin(assessment_stay_ids)]
cohort_flow_rows.append(build_cohort_counts('5. Delirium assessment available', assessment_cohort, assessments_all))

print(f"\nDelirium assessment rows after stay-level criteria: {len(assessments_all):,}")
print(f"  - Unique stays: {assessments_all['stay_id'].nunique()}")

# Exclude assessment times before 8 full hours are available
before_filter = len(assessments_all)
assessments = assessments_all[assessments_all['assessment_bin'] >= 7].copy()
print(f"assessment_bin < 7 excluded: {before_filter:,} -> {len(assessments):,} ({before_filter - len(assessments):,} excluded)")

window_stay_ids = set(assessments['stay_id'])
window_cohort = cohort_8hrs[cohort_8hrs['stay_id'].isin(window_stay_ids)]
cohort_flow_rows.append(build_cohort_counts('6. Assessment has full 8-hour window', window_cohort, assessments))

print(f"\nDelirium assessment rows: {len(assessments):,}")
print(f"  - Positive (1): {(assessments['Delirium'] == 1).sum():,}")
print(f"  - Negative (0): {(assessments['Delirium'] == 0).sum():,}")
print(f"  - Unique stays: {assessments['stay_id'].nunique()}")

cohort_flow = pd.DataFrame(cohort_flow_rows)
cohort_flow['stays_removed_from_previous'] = cohort_flow['n_stays'].shift(1).fillna(cohort_flow['n_stays']).astype(int) - cohort_flow['n_stays'].astype(int)
cohort_flow['stays_pct_of_initial'] = cohort_flow['n_stays'] / cohort_flow.loc[0, 'n_stays'] * 100
cohort_flow.to_csv(OUTPUT_DIR / 'cohort_attrition.csv', index=False)
display(cohort_flow)


In [ ]:
# 각 섬망 평가 시점마다 직전 8시간 window의 bin 목록을 만듭니다.
# assessment별 window 데이터를 timeseries와 결합해 집계 입력을 구성합니다.
# Build 8-hour window bins for each assessment
assessments['window_bins'] = assessments['assessment_bin'].apply(
    lambda b: list(range(max(0, int(b) - 7), int(b) + 1))
)
assessment_exploded = assessments.explode('window_bins')
assessment_exploded['bin'] = assessment_exploded['window_bins'].astype(int)
assessment_exploded.drop(columns='window_bins', inplace=True)

print(f"Exploded shape (assessments x window bins): {assessment_exploded.shape}")

window_data = assessment_exploded.merge(
    timeseries.drop(columns='Delirium'),
    on=['stay_id', 'bin'],
    how='inner'
)
print(f"Window data shape: {window_data.shape}")


In [ ]:
# 최종 모델 입력 변수를 집계 방식별로 분류하고 assessment 단위로 요약합니다.
# 활력징후는 평균/표준편차, binary 노출은 max, lab/신경학적 값은 최신값을 사용합니다.
vital_mean_std_cols = [
    'heart_rate', 'nibp_mbp', 'abp_mbp', 'spo2', 'sao2', 'temperature', 'respiratory_rate'
]
vital_mean_std_cols = [c for c in vital_mean_std_cols if c in window_data.columns]

medication_feature_cols = []
med_events_path = EXTRACT_DIR / 'medication_events.csv'
if med_events_path.exists():
    medication_feature_cols = pd.read_csv(med_events_path, usecols=['feature_name'])['feature_name'].dropna().unique().tolist()

binary_cols = [c for c in window_data.columns if c.startswith('procedure_')]
binary_cols += [c for c in medication_feature_cols if c in window_data.columns]
binary_cols = sorted(set(binary_cols))

latest_value_cols = [
    'weight', 'height',
    'rass', 'gcs_eye', 'gcs_verbal', 'gcs_motor',
    'wbc', 'hemoglobin', 'hematocrit', 'platelet_count',
    'sodium', 'potassium', 'chloride', 'bicarbonate', 'bicarbonate_gas',
    'calcium_free', 'magnesium', 'phosphate', 'anion_gap',
    'BUN', 'creatinine', 'alt', 'ast',
    'bilirubin_total', 'bilirubin_direct', 'bilirubin_indirect', 'albumin',
    'glucose', 'lactate', 'ph', 'pco2', 'po2', 'inr_pt', 'pt', 'ptt',
    'nibp_sbp', 'abp_sbp', 'nibp_dbp', 'abp_dbp'
]
latest_value_cols = [c for c in latest_value_cols if c in window_data.columns]

group_key = ['stay_id', 'assessment_bin', 'Delirium']

# 1. Demographics
demo_cols = [c for c in ['age', 'gender', 'los_hours', 'subject_id', 'hadm_id', 'admission_type', 'race', 'hospital_expire_flag'] if c in window_data.columns]
demo_agg = window_data.groupby(group_key)[demo_cols].first().reset_index()

# 2. V/S: 8-hour mean + std
vital_agg = window_data[group_key].drop_duplicates().copy()
if vital_mean_std_cols:
    vital_means = window_data.groupby(group_key)[vital_mean_std_cols].mean()
    vital_means.columns = [f'{c}_mean' for c in vital_means.columns]
    vital_stds = window_data.groupby(group_key)[vital_mean_std_cols].std()
    vital_stds.columns = [f'{c}_std' for c in vital_stds.columns]
    vital_agg = pd.concat([vital_means, vital_stds], axis=1).reset_index()

# 3. Medication/procedure: any exposure in the 8-hour window
binary_agg = window_data[group_key].drop_duplicates().copy()
if binary_cols:
    binary_agg = window_data.groupby(group_key)[binary_cols].max().reset_index()
    for col in binary_cols:
        binary_agg[col] = (binary_agg[col] > 0).astype(int)

# 4. Labs/body/neuro: latest value in the 8-hour window
latest_agg = window_data[group_key].drop_duplicates().copy()
if latest_value_cols:
    window_sorted = window_data.sort_values('bin', ascending=False)
    latest_agg = window_sorted.groupby(group_key)[latest_value_cols].first().reset_index()

# Merge
final_df = demo_agg.merge(vital_agg, on=group_key, how='left')
final_df = final_df.merge(binary_agg, on=group_key, how='left')
final_df = final_df.merge(latest_agg, on=group_key, how='left')
final_df = final_df.sort_values(['stay_id', 'assessment_bin']).reset_index(drop=True)

print("\n=== Final Dataset ===")
print(f"Shape: {final_df.shape}")
print(f"Unique stays: {final_df['stay_id'].nunique()}")
print("\nDelirium distribution:")
print(final_df['Delirium'].value_counts())
print(f"\nColumns ({len(final_df.columns)}):")
print(final_df.columns.tolist())


In [ ]:
# 최종 데이터셋의 결측률을 확인합니다.
# 남아 있는 결측 변수를 확인해 후속 모델링 전 처리 필요성을 점검합니다.
# Result check
print("=== Missingness ===")
missing_pct = final_df.isnull().sum() * 100 / len(final_df)
print(missing_pct[missing_pct > 0].sort_values(ascending=False).to_string())
if missing_pct.sum() == 0:
    print("No missing values")


In [ ]:
# 필요 시 변수를 수동으로 제외하는 셀입니다.
# 현재 Parkinson 분석에서는 기본적으로 제거하는 변수가 없도록 비워둡니다.
# Manual drop cell kept to match the reference notebook flow.
# For Parkinson, no variables are dropped by default.
drop_cols = []
final_df.drop(columns=[c for c in drop_cols if c in final_df.columns], inplace=True)
print(f"Dropped variables: {drop_cols}")
print(f"Shape after drop: {final_df.shape}")
print(f"Columns ({len(final_df.columns)}): {final_df.columns.tolist()}")


In [ ]:
# 최종 assessment 데이터셋과 시간별 timeseries 산출물을 저장합니다.
# 모델링용 final_dataset.csv와 재현용 중간 산출물을 함께 남깁니다.
# Save
final_df.to_csv(OUTPUT_DIR / 'final_dataset.csv', index=False)
final_df.to_csv(OUTPUT_DIR / 'assessment_dataset_60min.csv', index=False)

if 'cohort_8hr_stay_ids' in globals():
    timeseries_output = timeseries[timeseries['stay_id'].isin(cohort_8hr_stay_ids)].copy()
else:
    timeseries_output = timeseries.copy()
timeseries_output.to_csv(OUTPUT_DIR / 'hourly_timeseries_60min.csv', index=False)

print(f"Saved: final_dataset.csv ({len(final_df):,} rows, {final_df['stay_id'].nunique()} stays)")
print(f"Saved: assessment_dataset_60min.csv ({len(final_df):,} rows)")
print(f"Saved: hourly_timeseries_60min.csv ({len(timeseries_output):,} rows)")
